# Notebook 08 — Natural Selection and Fitness Models

**Module:** 05 — Biology Fundamentals  
**Notebook:** 08 of 18  
**Prerequisites:** NB07  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Why do some alleles become common and others disappear? Selection — differential
survival and reproduction based on genotype — is the answer. Every evolutionary
analysis in computational biology (detecting selective sweeps, estimating dN/dS
ratios, predicting protein fitness effects) depends on this model.

---
## Step 2 — Intuition

Think of fitness as a multiplier on how many offspring a genotype leaves. If genotype AA
leaves 1.1× as many offspring as aa, then A becomes more common each generation.
The rate of spread depends on both s (the fitness advantage) and the current allele
frequency. An allele at p=0.01 can barely 'feel' selection — most individuals don't
carry it. At p=0.5, selection drives fast change.

---
## Step 3 — Biological Background

**Types of natural selection:**

| Type | Effect | Example |
|------|--------|---------|
| **Directional** | favours one extreme → shifts mean | Antibiotic resistance |
| **Stabilising** | favours intermediate phenotype | Human birth weight |
| **Disruptive** | favours both extremes | Beak size in Darwin's finches |
| **Balancing** | maintains polymorphism | Sickle-cell / malaria |
| **Purifying** | removes harmful alleles | Deleterious mutations in coding regions |
| **Positive** | favours beneficial alleles | Lactase persistence in Europeans |

**The selection coefficient (s):**
Defines the relative fitness of each genotype. Using additive fitness:

| Genotype | Fitness |
|----------|---------|
| AA | 1 + 2s |
| Aa | 1 + s |
| aa | 1 |

Positive s → A is favoured; negative s → A is disfavoured (deleterious).

**Heterozygote advantage (overdominance):**
Sickle-cell anemia: aa causes sickle-cell disease; AA is susceptible to malaria;
Aa is protected from both. This balancing selection maintains both alleles in
malaria-endemic populations (Africa, Mediterranean).

**dN/dS ratio (Ka/Ks):**
- dN: rate of non-synonymous substitutions (amino acid-changing)
- dS: rate of synonymous substitutions (silent, amino acid-preserving)
- dN/dS < 1: purifying selection (most non-synonymous mutations are harmful)
- dN/dS = 1: neutral evolution
- dN/dS > 1: positive selection (non-synonymous changes are favoured)

---
## Step 4 — Mathematical Explanation

**Allele frequency change per generation (additive model):**

$$\bar{w} = p^2(1+2s) + 2pq(1+s) + q^2 = 1 + 2ps$$

After selection:
$$p' = \frac{p(1+s)(p + q)}{\bar{w}} = \frac{p + sp^2}{\bar{w}}$$

Change per generation:
$$\Delta p = p' - p = \frac{sp(1-p)}{1 + 2sp} \approx sp(1-p) \text{ for small } s$$

Note: Δp is largest when p = 0.5, slowest near p = 0 or p = 1.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Deterministic selection dynamics
def selection_trajectory(
    p0: float,
    s: float,
    n_gen: int,
    dominance: str = 'additive',
) -> np.ndarray:
    """
    Simulate deterministic allele frequency change under selection.

    dominance: 'additive', 'dominant' (A fully dominant), 'recessive' (a recessive)
    """
    p = p0
    trajectory = [p]

    for _ in range(n_gen):
        q = 1.0 - p
        if dominance == 'additive':
            # w_AA=1+2s, w_Aa=1+s, w_aa=1
            w_bar = p**2 * (1+2*s) + 2*p*q * (1+s) + q**2
            p_new = (p**2 * (1+2*s) + p*q * (1+s)) / w_bar
        elif dominance == 'dominant':
            # A is fully dominant: w_AA=w_Aa=1+s, w_aa=1
            w_bar = (p**2 + 2*p*q) * (1+s) + q**2
            p_new = ((p**2 + p*q) * (1+s)) / w_bar
        elif dominance == 'recessive':
            # A is fully recessive: w_AA=1+s, w_Aa=w_aa=1
            w_bar = p**2 * (1+s) + 2*p*q + q**2
            p_new = (p**2 * (1+s) + p*q) / w_bar
        p = np.clip(p_new, 0.0, 1.0)
        trajectory.append(p)

    return np.array(trajectory)

# Compare selection coefficients
gen = np.arange(1001)
for s_val in [0.001, 0.01, 0.05, 0.1]:
    traj = selection_trajectory(p0=0.01, s=s_val, n_gen=1000)
    # Find generation when p crosses 0.5
    cross = np.argmax(traj >= 0.5) if np.any(traj >= 0.5) else -1
    print(f"  s={s_val:.3f}: p reaches 0.5 at generation {cross if cross > 0 else '>1000'}")

In [ ]:
# Cell 6.2 — Overdominance (heterozygote advantage): stable equilibrium
def overdominance_equilibrium(s_AA: float, s_aa: float) -> float:
    """
    Equilibrium frequency under overdominance.
    w_AA = 1 - s_AA, w_Aa = 1, w_aa = 1 - s_aa
    Equilibrium: p* = s_aa / (s_AA + s_aa)
    """
    return s_aa / (s_AA + s_aa)

# Sickle cell: AA susceptible to malaria (s_AA≈0.15), aa has sickle-cell disease (s_aa≈0.9)
# in malaria-endemic regions
s_AA_malaria = 0.15
s_aa_sickle  = 0.90
p_star = overdominance_equilibrium(s_AA_malaria, s_aa_sickle)
print(f"Sickle-cell overdominance example:")
print(f"  s(AA) = {s_AA_malaria} (malaria susceptible), s(aa) = {s_aa_sickle} (sickle-cell disease)")
print(f"  Stable equilibrium frequency of A (normal allele): p* = {p_star:.3f}")
print(f"  Carrier frequency at equilibrium:                 2pq = {2*p_star*(1-p_star):.3f}")

In [ ]:
# Cell 6.3 — Purifying selection removes deleterious alleles
# Simulate trajectory of a deleterious allele (s<0) over generations
purifying = selection_trajectory(p0=0.5, s=-0.05, n_gen=500)
positive  = selection_trajectory(p0=0.01, s=0.05, n_gen=500)

print(f"Purifying selection (s=-0.05, p0=0.5): final p = {purifying[-1]:.4f}")
print(f"Positive selection  (s=+0.05, p0=0.01): final p = {positive[-1]:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Selection trajectories + Δp curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Selection coefficient comparison
ax = axes[0]
colors_s = ['#d4edda', '#b3d9ff', '#fff3cd', '#f8d7da']
for s_val, color in zip([0.001, 0.01, 0.05, 0.10], colors_s):
    traj = selection_trajectory(p0=0.01, s=s_val, n_gen=1000)
    ax.plot(np.arange(1001), traj, label=f's={s_val}', lw=2, color=color,
            mec='gray')

# Compare additive vs. dominant vs. recessive for s=0.05
for dom, ls in [('dominant','--'), ('recessive',':')]:
    traj = selection_trajectory(p0=0.01, s=0.05, n_gen=1000, dominance=dom)
    ax.plot(np.arange(1001), traj, ls=ls, color='gray', alpha=0.6, label=f's=0.05 ({dom})')

ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency p')
ax.set_title('Positive selection: rate of allele frequency increase')
ax.legend(fontsize=7)

# Panel 2: Δp as function of p (shows where selection is fastest)
ax = axes[1]
p_vals = np.linspace(0.01, 0.99, 200)
for s_val, color in zip([0.01, 0.05, 0.10], ['steelblue', 'tomato', 'seagreen']):
    delta_p = s_val * p_vals * (1 - p_vals)  # approximate Δp
    ax.plot(p_vals, delta_p, label=f's={s_val}', color=color, lw=2)
ax.set_xlabel('Current allele frequency p')
ax.set_ylabel('Δp per generation (approximate)')
ax.set_title('Rate of allele frequency change:\nfastest when p ≈ 0.5')
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `selection_trajectory(p0, s, n_gen)` from scratch using the additive
   fitness model. Verify that with s=0, p stays constant.
2. The 'selective sweep' signature in population genetics is low genetic diversity
   near a selected allele. Why? (Think about what happens to nearby alleles when
   a beneficial allele rises rapidly to fixation.)
3. dN/dS < 1 for most genes. What does this tell you about the nature of most
   coding mutations? Name one gene family where you would expect dN/dS > 1.
4. A new drug-resistant allele in a bacterial population has s = 0.1 relative to
   drug-sensitive bacteria, and starts at p0 = 0.001. Using your simulation,
   how many generations until it reaches p = 0.99?

---
## Quiz — Active Recall

1. What is the selection coefficient s? What does s=0.05 mean biologically?
2. At what allele frequency is the rate of change due to selection fastest?
3. What is heterozygote advantage? Give the sickle-cell example.
4. What does dN/dS > 1 indicate about a gene's evolutionary history?
5. What is purifying selection? Why are most coding mutations subject to it?

---
## Reflection

**Date completed:** ____________________

> *[An antibiotic is introduced to a hospital. Resistance is caused by a single
> allele, currently at 0.1% frequency. If s=0.3 (strong advantage under antibiotics),
> is the hospital in trouble? Simulate it and answer.]*

---
*Next: `09_speciation_and_phylogenetics.ipynb`*